Procesado de datos de las baterías 13300 para el TFG de reciclado de baterías de vape.

Primero analizaremos los ciclos y veremos el SoH de las baterías

In [9]:
import os
import shutil
import re
import glob


def organizar_archivos_por_fecha(carpeta_origen):
    """
    Lee los archivos de una carpeta, extrae la fecha de su nombre 
    y los mueve a subcarpetas organizadas por día.
    """
    
    # Expresión regular para buscar el formato de fecha AAAA_MM_DD
    # \d{4} = 4 dígitos (año), \d{2} = 2 dígitos (mes/día)
    patron_fecha = re.compile(r"(\d{4}_\d{2}_\d{2})")

    # Verificamos que la carpeta de origen exista
    if not os.path.exists(carpeta_origen):
        print(f"Error: La carpeta '{carpeta_origen}' no existe.")
        return

    # Iterar sobre todos los elementos en la carpeta
    for nombre_archivo in os.listdir(carpeta_origen):
        ruta_completa_origen = os.path.join(carpeta_origen, nombre_archivo)

        # Nos aseguramos de procesar solo archivos (ignoramos carpetas)
        if os.path.isfile(ruta_completa_origen):
            
            # Buscamos el patrón de la fecha en el nombre del archivo
            coincidencia = patron_fecha.search(nombre_archivo)
            
            if coincidencia:
                # Extraemos la fecha encontrada (ej. '2026_02_23')
                fecha_carpeta = coincidencia.group(1) 
                
                # Definimos la ruta de la nueva carpeta destino
                ruta_carpeta_destino = os.path.join("data/processed/cicladores", str(i), fecha_carpeta)
                
                # Si la carpeta de esa fecha no existe, la creamos
                if not os.path.exists(ruta_carpeta_destino):
                    os.makedirs(ruta_carpeta_destino)
                
                # Movemos el archivo
                ruta_completa_destino = os.path.join(ruta_carpeta_destino, nombre_archivo)
                shutil.copy(ruta_completa_origen, ruta_completa_destino)
                
                #print(f"✅ Copiado: {nombre_archivo}  ->  Carpeta: {fecha_carpeta}/")
            else:
                print(f"⚠️ Omitido (sin fecha detectada): {nombre_archivo}")

def recortar_archivo_txt(ruta_archivo):
    """
    Lee un archivo de texto y lo recorta, dejando únicamente
    las primeras 2 líneas y las últimas 10 líneas.
    """
    # Verificamos que el archivo exista
    if not os.path.exists(ruta_archivo):
        print(f"Error: No se encontró el archivo {ruta_archivo}")
        return

    # Leemos todas las líneas del archivo
    with open(ruta_archivo, 'r', encoding='utf-8') as archivo:
        lineas = archivo.readlines()

    # Si el archivo tiene 12 líneas o menos, no tiene sentido recortarlo 
    # (evitamos duplicar líneas)
    if len(lineas) <= 12:
        print(f"⚠️ El archivo tiene 12 líneas o menos, no se requiere recorte: {ruta_archivo}")
        return

    # Juntamos las primeras 2 y las últimas 10 líneas
    lineas_recortadas = lineas[:2] + lineas[-10:]

    # Escribimos las líneas seleccionadas sobrescribiendo el archivo original
    with open(ruta_archivo, 'w', encoding='utf-8') as archivo:
        archivo.writelines(lineas_recortadas)

    print(f"✅ Archivo recortado con éxito: {os.path.basename(ruta_archivo)}")


In [13]:

for i in range(1, 7):

    mi_carpeta = f"data/raw/cicladores/ciclador_{i}"
    
    print(f"Procesando: {mi_carpeta} ...")
    
    # Llamamos a la función con la ruta actualizada
    organizar_archivos_por_fecha(mi_carpeta)

print("¡Todas las carpetas han sido organizadas!")

recortar_archivo_txt("data/processed/cicladores/6/2026_02_23/STP11_Charge 1C Elias_2026_02_23_17_12_50.txt")

# Iteramos sobre los 6 cicladores
for j in range(1, 7):
    carpeta = f"data/processed/cicladores/{j}"
    
    # Buscamos todos los archivos .txt dentro de esa carpeta
    archivos_txt = glob.glob(os.path.join(carpeta, "*.txt"))
    
    for archivo in archivos_txt:
        recortar_archivo_txt(archivo)

Procesando: data/raw/cicladores/ciclador_1 ...
Procesando: data/raw/cicladores/ciclador_2 ...
Procesando: data/raw/cicladores/ciclador_3 ...
Procesando: data/raw/cicladores/ciclador_4 ...
Procesando: data/raw/cicladores/ciclador_5 ...
Procesando: data/raw/cicladores/ciclador_6 ...
¡Todas las carpetas han sido organizadas!


UnicodeDecodeError: 'utf-8' codec can't decode byte 0xb0 in position 329: invalid start byte

Ya teniendo cada ciclador ordenado pr día vamos a casar cada archivo con cada batería
Usaremos la info de tiempo y el xlsx que tenemos.